# AIR Demo — Bob (CHSH rounds)

Reconciles Bob's sifted **CHSH Bell-test** bits (from `data/data4eat-sep/`)
with Alice using the separate-IR AIR protocol: pure-shortening polar code at
N = 2^15, SCL = 32.

- Assumed QBER: 0.1825 (95% two-sided Wald upper bound on the CHSH calibration
  BER, ceiled to 5e-4)
- Three AIR stages: f = 1.05, f = 1.10, then a union-bound stage targeting
  99.999% success probability
- Pure shortening: S = 12871 of N = 32768, P = 0

Run **this notebook first**, then Alice's (`qkd-air-demo-alice-chsh.ipynb`).

In [ ]:
import sys
import os
import time
import gc
from datetime import datetime
from copy import deepcopy

sys.path.insert(0, '../../submodules/py_aff3ct/build/lib')
import numpy as np
import py_aff3ct

sys.path.insert(0, '../../src/')
from utils import (
    generate_chsh_flip_mask_from_basis,
    generate_shortened_from_file, generate_punctured_from_file,
    generate_shortened_BGL, generate_punctured_BGL,
    generate_frozen_position_from_file,
    num_frozen_bits_from_union_bound_from_file,
    llr_calculater_bsc,
    mask_fillor, extractor,
    NumpyArrayHttpSender, NumpyArrayHttpReceiver,
)
from py_aff3ct.module.encoder import Encoder_polar
from py_aff3ct.module.decoder import Decoder_polar_SCL_naive_CA
from py_aff3ct.module.crc import CRC_polynomial


## Load experimental data

Bob needs his own data plus `alice_basis` (public), `alice_result` and
`snspd_click` for the SNSPD correction and the per-subgroup CHSH flip, and
selects the CHSH Bell-test rounds with his own round classification
(`exp_type == 0` in the sifting string he publishes).

In [ ]:
DATA_DIR = '../../data/data4eat-sep/'

bob_result_raw = np.load(os.path.join(DATA_DIR, 'bob_result.npy')).astype(np.int8)
bob_basis      = np.load(os.path.join(DATA_DIR, 'bob_basis.npy'))
bob_exp_type   = np.load(os.path.join(DATA_DIR, 'bob_exp_type.npy'))
alice_basis    = np.load(os.path.join(DATA_DIR, 'alice_basis.npy'))
alice_result   = np.load(os.path.join(DATA_DIR, 'alice_result.npy')).astype(np.int8)
snspd_click    = np.load(os.path.join(DATA_DIR, 'snspd_click.npy'))

# Keep only CHSH Bell-test rounds (exp_type == 0 in the sifting string),
# same selection as ir-raw-chsh.py
chsh_mask = (bob_exp_type == 0)
num_total = int(chsh_mask.sum())
print(f'Total rounds: {len(bob_result_raw)}, CHSH rounds: {num_total}')


## Three-step bit-flip pipeline (Bob's side)

Bob applies **steps 1–2**:
- SNSPD correction: flip where `alice_basis == 1 & snspd_click == 1`
- Per-subgroup CHSH flip: flip the E > 0 subgroups of the calibration
  data (`generate_chsh_flip_mask_from_basis`)

Alice applies step 3 (AIR flip) on her side.

In [ ]:
bob_result = bob_result_raw[chsh_mask].copy()

flip_snspd = (alice_basis[chsh_mask] == 1) & (snspd_click[chsh_mask] == 1)
n_snspd = int(flip_snspd.sum())
if n_snspd > 0:
    bob_result[flip_snspd] = 1 - bob_result[flip_snspd]
print(f'SNSPD-corrected positions: {n_snspd}')

FLIP_SUBGROUPS = [(0, 1), (1, 0), (1, 1)]  # E > 0 subgroups of the calibration data
chsh_flip_mask = generate_chsh_flip_mask_from_basis(
    alice_basis[chsh_mask], bob_basis[chsh_mask], FLIP_SUBGROUPS,
)
n_chsh_flip = int(chsh_flip_mask.sum())
if n_chsh_flip > 0:
    bob_result[chsh_flip_mask] = 1 - bob_result[chsh_flip_mask]
print(f'CHSH-flipped positions: {n_chsh_flip} (calibration subgroups)')

result_ir = bob_result.astype(np.int32)
len_sifted_key = result_ir.shape[0]
print(f'Sifted key length: {len_sifted_key}')


## Configure parameters

Same parameters as Alice's notebook — they MUST match for the protocol to
work: assumed QBER = 0.1825, SCL = 32, CRC = 32, pure shortening
S = N − L = 12871, P = 0, and the same three AIR stages
(1.05 / 1.10 / union-bound @ 99.999% success).

In [ ]:
# --- Assumed QBER -----------------------------------------------------
# 0.1825 = 95% two-sided Wald upper bound on the CHSH calibration BER
# (point estimate 0.172920, sigma 0.004883, upper 0.182492), ceiled to 5e-4.
# The polar-code reliability file n15_sp-bsc_ber0.1825_mu100_*.pc exists at
# this exact BER.
assumed_q_ber = 0.1825

# Real QBER, computed from the (locally available) dataset after the 3-step
# flip pipeline -- used only for reporting the achieved efficiency.
# (Mirrors ir-raw-chsh.py: SNSPD + per-subgroup CHSH flips on Bob, AIR flip
# on Alice. In a real deployment this would come from calibration.)
alice_result_f = np.load(os.path.join(DATA_DIR, 'alice_result.npy')).astype(np.int8)
bob_result_raw_f = np.load(os.path.join(DATA_DIR, 'bob_result.npy')).astype(np.int8)
alice_basis_f = np.load(os.path.join(DATA_DIR, 'alice_basis.npy'))
bob_basis_f = np.load(os.path.join(DATA_DIR, 'bob_basis.npy'))
snspd_click_f = np.load(os.path.join(DATA_DIR, 'snspd_click.npy'))
alice_ir = (1 - alice_result_f[chsh_mask]).astype(np.int32)
bob_c = bob_result_raw_f[chsh_mask].copy()
_flip = (alice_basis_f[chsh_mask] == 1) & (snspd_click_f[chsh_mask] == 1)
bob_c[_flip] = 1 - bob_c[_flip]
_flip2 = generate_chsh_flip_mask_from_basis(
    alice_basis_f[chsh_mask], bob_basis_f[chsh_mask], FLIP_SUBGROUPS)
bob_c[_flip2] = 1 - bob_c[_flip2]
q_ber_stat = float(np.logical_xor(alice_ir, bob_c.astype(np.int32)).sum() / len_sifted_key)
print(f'Real BER (after 3-step flip): {q_ber_stat:.6f}')

shannon_H2 = lambda x: -x * np.log2(x) - (1 - x) * np.log2(1 - x) if 0 < x < 1 else 0.0

n = int(np.ceil(np.log2(len_sifted_key)))
block_encoded_bits = 2 ** n
scl_len = 32
crc_size = 32

# Pure shortening: S = N - L, P = 0 (configuration mirrors ir-raw-chsh.py)
S_BITS = block_encoded_bits - len_sifted_key
P_BITS = 0
assert S_BITS == 12871, \
    f'S_BITS = {S_BITS} != 12871: the sp/pe reliability files were generated for S = 12871'

ideal_frozen_bits = len_sifted_key * shannon_H2(q_ber_stat)
assumed_ideal_frozen_bits = len_sifted_key * shannon_H2(assumed_q_ber)
crc_efficiency_gain = crc_size / assumed_ideal_frozen_bits

print(f'N={block_encoded_bits}, L={len_sifted_key}, S={S_BITS}, P={P_BITS}')
print(f'Assumed QBER={assumed_q_ber}, real QBER={q_ber_stat:.6f}')


## Shortened / punctured positions (pure shortening)

Identical to Alice's side (S = 12871, P = 0): positions are loaded from
`conf/polar/sp-bsc/shortened_positions_N=15_S=12871.txt` when available
(generated by polar-fbgen-rust), else recomputed via BGL.

In [ ]:
PC_DIR = '../../conf/polar/sp-bsc/'

sp_file_S = os.path.join(PC_DIR, f'shortened_positions_N={n}_S={S_BITS}.txt')
sp_file_P = os.path.join(PC_DIR, f'punctured_positions_N={n}_P={P_BITS}.txt')

if os.path.exists(sp_file_S):
    shortened_idx = generate_shortened_from_file(sp_file_S)
    print(f'Shortened positions loaded from file: {len(shortened_idx)} positions')
else:
    shortened_idx = generate_shortened_BGL(block_encoded_bits, S_BITS)
    print(f'Shortened positions generated via BGL: {len(shortened_idx)} positions')

if os.path.exists(sp_file_P):
    punctured_idx = generate_punctured_from_file(sp_file_P)
    print(f'Punctured positions loaded from file: {len(punctured_idx)} positions')
else:
    punctured_idx = generate_punctured_BGL(block_encoded_bits, P_BITS)
    print(f'Punctured positions generated via BGL: {len(punctured_idx)} positions')

# Sanity: disjoint
overlap = set(shortened_idx.tolist()) & set(punctured_idx.tolist())
assert len(overlap) == 0, f'Shortened and punctured sets overlap at {len(overlap)} positions'

shortened_vec = np.zeros(block_encoded_bits, dtype=bool)
shortened_vec[shortened_idx] = True
punctured_vec = np.zeros(block_encoded_bits, dtype=bool)
punctured_vec[punctured_idx] = True
sp_vec = shortened_vec | punctured_vec
sp_idx = np.concatenate([shortened_idx, punctured_idx])

non_shortened_mask = (~sp_vec).astype(np.int32).reshape((1, -1))
non_sp_1d = np.where(non_shortened_mask.flatten() == 1)[0]
assert len(non_sp_1d) == len_sifted_key, \
    f'non-sp count {len(non_sp_1d)} != L {len_sifted_key}'
print(f'  Transmitted positions: {len(non_sp_1d)} (== L)')


## AIR efficiency stages and frozen vectors

Identical to Alice's side: f = 1.05, f = 1.10, and a union-bound stage
targeting **99.999% success** probability (failure rate ε = 1e-5).

In [ ]:
# --- AIR stages -------------------------------------------------------
# 1. f = 1.05 (configured efficiency vs. assumed ideal)
# 2. f = 1.10
# 3. f from the union bound on the frozen_bits PE file, targeting 99.999%
#    success probability (failure rate eps = 1e-5)
pe_file = os.path.join(
    PC_DIR,
    f'frozen_bits_mu=100_N={n}_p={assumed_q_ber}_shortened={S_BITS}_punctured={P_BITS}.txt',
)
assert os.path.exists(pe_file), f'PE file not found: {pe_file}'

kf_ub = num_frozen_bits_from_union_bound_from_file(pe_file, sp_idx, 1e-5)
eff_ub = kf_ub / assumed_ideal_frozen_bits
configured_efficiency_stage = np.array([1.05, 1.10, eff_ub], dtype=np.float64)

# Subtract the CRC cost before converting efficiencies to frozen-bit counts,
# exactly as in air-mc-keygen-calibration.py / ir-raw-chsh.py.
efficiency_stage = configured_efficiency_stage - crc_efficiency_gain
effective_frozen_bits_stage = np.asarray(efficiency_stage * assumed_ideal_frozen_bits, dtype=np.int64)
effective_efficiency_stage = effective_frozen_bits_stage / assumed_ideal_frozen_bits
info_bits_stage = block_encoded_bits - effective_frozen_bits_stage - S_BITS - P_BITS
# Decoder information positions: AIR information (P = 0: no free punctured leaves)
dec_info_bits_stage = block_encoded_bits - effective_frozen_bits_stage - S_BITS
assert (info_bits_stage > 0).all(), f'info_bits must be > 0, got {info_bits_stage}'
assert np.all(np.diff(effective_frozen_bits_stage) > 0), 'frozen bits must be strictly increasing'

print(f'Union-bound frozen bits @99.999% success: {kf_ub} (eff = {eff_ub:.6f})')
print(f'Configured efficiency stages: {configured_efficiency_stage.tolist()}')
print(f'Effective frozen bits       : {effective_frozen_bits_stage.tolist()}')
print(f'Info bits                   : {info_bits_stage.tolist()}')

# --- Frozen vectors per stage -----------------------------------------
fb_file_dir = os.path.join(
    PC_DIR,
    f'n{n}_sp-bsc_ber{assumed_q_ber}_mu100_shortened{S_BITS}_punctured{P_BITS}.pc',
)
assert os.path.exists(fb_file_dir), f'.pc file not found: {fb_file_dir}'
print(f'Loading polar code: {fb_file_dir}')

frozen_vec_stage = []
effective_frozen_vec_stage = []
for efb in effective_frozen_bits_stage:
    _, _, eff_frozen_idx = generate_frozen_position_from_file(
        fb_file_dir, int(efb), sp_idx, verbose=False)
    # Decoder-frozen = shortened + effective (pure shortening: no free punctured leaves)
    fv = shortened_vec.copy()
    fv[eff_frozen_idx] = True
    ev = np.zeros(block_encoded_bits, dtype=bool)
    ev[eff_frozen_idx] = True
    frozen_vec_stage.append(fv)
    effective_frozen_vec_stage.append(ev)

effective_frozen_vec_mask_stage = [ev.astype(np.int32).reshape((1, -1)) for ev in effective_frozen_vec_stage]
print(f'{len(frozen_vec_stage)} stages prepared')


## Build LLR pipeline

Bob converts his padded key to BSC LLRs at the assumed BER and feeds
them directly to the decoder — the same chain as `ir-raw-chsh.py`. The
decoder amplifies the LLR at the known-zero shortened positions
internally (set_known_bits); shortening makes them an exact hard
constraint on the codeword side.

In [ ]:
non_shortened_mask = (~sp_vec).astype(np.int32).reshape((1, -1))

llr_bsc = llr_calculater_bsc(assumed_q_ber, block_encoded_bits)
# The known-zero padding amplification (shortened positions, x17) happens inside
# the decoder via its 'set_known_bits' task -- see the decode cell below.
known_bits_mask = sp_vec.astype(np.int32).reshape((1, -1))

padder_bob = mask_fillor(len_sifted_key, block_encoded_bits, dtype=np.int32)
fixed_bob = result_ir.reshape((1, -1))
padded_bob = np.zeros((1, block_encoded_bits), dtype=np.int32)
padder_bob['fill ::output'].bind(padded_bob)
padder_bob['fill ::mask'].bind(non_shortened_mask)
padder_bob['fill ::input'].bind(fixed_bob)
padder_bob['fill'].exec()

bob_decode_llr = np.zeros((1, block_encoded_bits), dtype=np.float32)
llr_bsc['llr_bsc ::input'].bind(padded_bob)
llr_bsc['llr_bsc ::output'].bind(bob_decode_llr)
llr_bsc['llr_bsc'].exec()

sp_llr = bob_decode_llr
print(f'LLR computed raw (padding amplification happens in the decoder: S={S_BITS})')


## Set up communication and logging

Bob sends on port 8012 and listens on port 8003 (Alice mirrors the roles).
Logs go to `log/eat-diqkd-sep/ir_demo_chsh/`.

In [ ]:
role = 'bob'
SENDER_PORT = 8012
RECEIVER_PORT = 8003
log_dir = '../../log/eat-diqkd-sep/ir_demo_chsh'
os.makedirs(log_dir, exist_ok=True)
start_time = datetime.now()
log_file = os.path.join(log_dir, f'qkd_air_demo_{role}_chsh_assumed_{assumed_q_ber}_{start_time.strftime("%Y%m%d_%H%M%S")}.txt')

class Logger:
    def __init__(self, log_file):
        self.log_file = log_file
        self.terminal = sys.stdout
    def write(self, message):
        self.terminal.write(message)
        with open(self.log_file, 'a') as f:
            f.write(message)
    def flush(self):
        self.terminal.flush()
        with open(self.log_file, 'a') as f:
            f.flush()

sys.stdout = Logger(log_file)
print(f'Log file: {log_file}')

sender_host = 'localhost'
sender_port = SENDER_PORT
receiver_host = 'localhost'
receiver_port = RECEIVER_PORT
array_sender = NumpyArrayHttpSender(host=sender_host, port=sender_port)
array_receiver = NumpyArrayHttpReceiver(host=receiver_host, port=receiver_port)
array_receiver.start_server()
print('HTTP communication ready')


## Run reconciliation

Bob waits for the start signal, receives the CRC tag, then for each stage
receives the newly appended frozen bits, accumulates them, and decodes with
a CA-SCL decoder (SCL = 32). One decoder is built per stage and freed
immediately after (~0.5 GB each at N = 2^15; a stage decode takes a few
seconds — 2.5 s in the committed demo log). The CRC status is sent back to Alice.

In [ ]:
num_bits_transmitted = 0
print('# ' * 16)
print(f'({datetime.now() - start_time}) Reconciliation Phase — Bob (CHSH)')
print('# ' * 16)

print('Bob: Waiting for Alice to start...')
http_start = time.time()
while array_receiver.received_array is None and (time.time() - http_start) < 1200:
    time.sleep(0.1)
if array_receiver.received_array is not None:
    array_receiver.received_array = None
    print('Bob: Received start signal')
else:
    print('Bob: Timeout waiting for start signal')

print(f'\n({datetime.now() - start_time}) Bob: Waiting for CRC tag...')
http_start = time.time()
while array_receiver.received_array is None and (time.time() - http_start) < 600:
    time.sleep(0.1)
if array_receiver.received_array is not None:
    bob_crc_tag = array_receiver.received_array.copy()
    array_receiver.received_array = None
else:
    print('Bob: Timeout waiting for CRC tag')
    bob_crc_tag = None

num_bits_transmitted += crc_size
print(f'CRC tag received: {bob_crc_tag}')

bob_decode_frozen_vector = np.zeros((1, block_encoded_bits), dtype=np.int32)
enc_bob = Encoder_polar(int(dec_info_bits_stage[0]), block_encoded_bits, frozen_vec_stage[0])
enc_bob.set_frozen_bits(frozen_vec_stage[0])
decoded_U_hat = np.zeros((1, block_encoded_bits), dtype=np.int32)
corrected_key = None

failure_flag = True
for stage in range(len(frozen_vec_stage)):
    print(f'\n({datetime.now() - start_time}) Stage {stage+1}/{len(frozen_vec_stage)}')

    print(f'Bob: Waiting for appended frozen bits...')
    http_start = time.time()
    while array_receiver.received_array is None and (time.time() - http_start) < 600:
        time.sleep(0.1)
    if array_receiver.received_array is not None:
        appended_bits = array_receiver.received_array.copy()
        array_receiver.received_array = None
    else:
        print('Bob: Timeout waiting for appended bits')
        break

    num_appended = appended_bits.shape[-1]
    num_bits_transmitted += num_appended
    print(f'Bob: Received {num_appended} bits')

    if stage == 0:
        indices = np.where(effective_frozen_vec_mask_stage[stage] == 1)
    else:
        indices = np.where(np.logical_xor(
            effective_frozen_vec_mask_stage[stage],
            effective_frozen_vec_mask_stage[stage-1]).astype(np.int32) == 1)
    bob_decode_frozen_vector[indices] = appended_bits

    # One CA-SCL decoder at a time: build, decode, free before the next stage
    # (same pattern as ir-raw-chsh.py).
    crc_dec = CRC_polynomial(block_encoded_bits, '0x04C11DB7', crc_size)
    dec = Decoder_polar_SCL_naive_CA(
        int(dec_info_bits_stage[stage]), block_encoded_bits, scl_len,
        frozen_vec_stage[stage], crc_dec)
    dec.set_frozen_bits(frozen_vec_stage[stage])
    dec['set_known_bits ::KNOWN_BITS'].bind(known_bits_mask)
    dec['set_known_bits'].exec()
    if bob_crc_tag is not None:
        dec['set_crc_const ::V_crc'].bind(bob_crc_tag)
        dec['set_crc_const'].exec()

    dec_status = np.ones((1, 1), dtype=np.int32)
    decoded_bob = np.zeros((1, block_encoded_bits), dtype=np.int32)
    dec['decode_siho_cw_flexible_frozen ::F_N'].bind(bob_decode_frozen_vector)
    dec['decode_siho_cw_flexible_frozen ::Y_N'].bind(sp_llr)
    dec['decode_siho_cw_flexible_frozen ::status'].bind(dec_status)
    dec['decode_siho_cw_flexible_frozen ::V_N'].bind(decoded_bob)

    t0 = time.time()
    dec['decode_siho_cw_flexible_frozen'].exec()
    dt = time.time() - t0
    print(f'Bob: decode {dt:.1f}s, status={int(dec_status.flatten()[0])}')

    # Recover the U-side estimate: the polar generator is involutory over
    # GF(2), so re-encoding the decoded codeword yields the decoded U. The
    # corrected key = U at effective (non-shortened, non-punctured)
    # positions -- saved regardless of CRC success.
    enc_bob['light_encode ::U_N'].bind(decoded_bob)
    enc_bob['light_encode ::X_N'].bind(decoded_U_hat)
    enc_bob['light_encode'].exec()
    corrected_key = decoded_U_hat.flatten()[non_sp_1d].copy()

    del dec, crc_dec, decoded_bob
    gc.collect()

    time.sleep(5)
    array_sender.send_array(dec_status)
    num_bits_transmitted += 1

    if np.abs(dec_status).sum() == 0:
        failure_flag = False
        print(f'Bob: CRC PASS at stage {stage+1}')
        print(f'Assumed efficiency: {num_bits_transmitted/assumed_ideal_frozen_bits:.6f}')
        print(f'Real efficiency:    {num_bits_transmitted/ideal_frozen_bits:.6f}')
        break
    else:
        print(f'Bob: CRC FAIL at stage {stage+1}')

if failure_flag:
    print(f'\n({datetime.now() - start_time}) All stages failed')


## Save corrected key

Save Bob's corrected key — the decode output of the **last attempted
stage**, restricted to effective (non-shortened, non-punctured)
positions, regardless of whether the CRC check passed.

In [ ]:
if corrected_key is not None:
    bob_out_file = os.path.splitext(log_file)[0] + '_bob_corrected.npy'
    np.save(bob_out_file, corrected_key)
    print(f'Bob corrected key saved: {bob_out_file} ({corrected_key.shape[0]} bits)')
else:
    print('No decode output to save')


## Cleanup

In [ ]:
del llr_bsc, padder_bob
del padded_bob, bob_decode_llr, sp_llr
del frozen_vec_stage, effective_frozen_vec_stage, effective_frozen_vec_mask_stage
del bob_decode_frozen_vector, enc_bob, decoded_U_hat
gc.collect()

array_receiver.stop_server()
print(f'\nTotal elapsed: {datetime.now() - start_time}')
print(f'Log file: {log_file}')
